# 03 · Evaluación semana 2 en Kaggle (GPU)
Primero ejecuta el benchmark de cinco folds LOSO para ambas tareas. Después cambia `MODE` a `run` y procesa rangos de folds reanudables.

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, zipfile, kagglehub
assert Path('/kaggle/working').exists() and os.environ.get('KAGGLE_KERNEL_RUN_TYPE')
REPO = Path('/kaggle/working/fpsd-cnn-reassessment')
if not (REPO/'pyproject.toml').is_file():
    candidates = [p.parent for p in Path('/kaggle/input').glob('**/pyproject.toml') if (p.parent/'src/fpsd_reassessment').is_dir()]
    if len(candidates) == 1: shutil.copytree(candidates[0], REPO, dirs_exist_ok=True)
    else:
        code_data = Path(kagglehub.dataset_download('sarahvasquez97/fpsd-cnn-reassessment-code'))
        archives = list(code_data.glob('*.zip'))
        if len(archives) == 1:
            REPO.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(archives[0]) as archive: archive.extractall(REPO)
        elif (code_data/'pyproject.toml').is_file(): shutil.copytree(code_data, REPO, dirs_exist_ok=True)
        else: raise FileNotFoundError(f'Paquete de código inválido: {code_data}')
feature_candidates = [p.parent for p in Path('/kaggle/input').glob('**/paper95_features.npz') if (p.parent/'windows_metadata.parquet').is_file()]
assert len(feature_candidates) == 1, feature_candidates
FEATURES = feature_candidates[0]
sys.path.insert(0, str(REPO/'src'))
os.environ['PYTHONPATH'] = str(REPO/'src')
print('Repositorio:', REPO, 'Características:', FEATURES)

In [ ]:
import torch, pandas as pd
assert torch.cuda.is_available(), 'Active una GPU en Notebook options → Accelerator.'
metadata = pd.read_parquet(FEATURES/'windows_metadata.parquet')
counts = metadata.drop_duplicates('subject_id').diagnosis.value_counts().to_dict()
assert counts == {'AD': 36, 'CN': 29, 'FTD': 23}, counts
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=str(REPO), check=True)
print(torch.cuda.get_device_name(0), counts)

In [ ]:
# Primera versión recomendada: MODE='benchmark'. Luego use MODE='run'.
MODE = 'benchmark'
TASK = 'ad_cn'            # ad_cn | cn_ftd
EVALUATION = 'loso'       # published_holdout | grouped5 | loso
BALANCE = 'none'          # none | subject_class
SEEDS = [2026]            # LOSO comienza con una sola semilla
FOLD_START = 0            # inclusivo
FOLD_STOP = 5             # exclusivo; ajustar por versión
OUTPUT = Path('/kaggle/working/week2_results')

In [ ]:
base = [sys.executable, str(REPO/'scripts/train_week2.py'), '--features', str(FEATURES), '--config', str(REPO/'configs/experiments/week2_subject_evaluation.yaml'), '--output', str(OUTPUT)]
if MODE == 'benchmark':
    for task in ('ad_cn', 'cn_ftd'):
        subprocess.run(base + ['--task', task, '--evaluation', 'loso', '--balance', 'none', '--benchmark-only', '--benchmark-folds', '5', '--fold-start', '0'], cwd=str(REPO), check=True)
elif MODE == 'run':
    command = base + ['--task', TASK, '--evaluation', EVALUATION, '--balance', BALANCE, '--fold-start', str(FOLD_START), '--fold-stop', str(FOLD_STOP), '--resume']
    for seed in SEEDS: command += ['--seed', str(seed)]
    subprocess.run(command, cwd=str(REPO), check=True)
else:
    raise ValueError(MODE)

In [ ]:
subprocess.run([sys.executable, str(REPO/'scripts/export_week2_results.py'), '--source', str(OUTPUT), '--output-dir', '/kaggle/working'], cwd=str(REPO), check=True)
print('Descargue week2_results_bundle.zip y week2_github_results_bundle.zip')